# Ring-Down Analysis from Arrays

This notebook demonstrates how to analyze ring-down data directly from numpy arrays or pandas Series/DataFrames using `RingDownAnalyzer.analyze_array()`, without loading from files.

We generate synthetic noisy ring-down data and run the full analysis pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ringdownanalysis import RingDownAnalyzer, RingDownSignal

In [ ]:
# Apply consistent plotting style
from ringdownanalysis import plots

plots.apply_plotting_style()

## Generate Synthetic Noisy Ring-Down Data

We use `RingDownSignal` to generate a noisy exponentially decaying sinusoid:
$$x(t) = A_0 \exp(-t/\tau) \cos(2\pi f_0 t + \phi_0) + \text{noise}$$

In [ ]:
# Signal parameters
f0 = 5.0  # Hz
fs = 1000.0  # Hz
N = 100000  # samples
A0 = 0.1
snr_db = 50.0  # dB
Q = 500.0  # quality factor

# Generate noisy ring-down signal
rng = np.random.default_rng(42)
signal = RingDownSignal(f0=f0, fs=fs, N=N, A0=A0, snr_db=snr_db, Q=Q)
t, data, phi0 = signal.generate(rng=rng)

print(f"Generated {N} samples at {fs} Hz")
print(f"True: f0={f0} Hz, tau={signal.tau:.4f} s, Q={Q}")
print(f"Noise sigma: {signal.sigma:.6f}")

## Analyze with NumPy Arrays

Pass time and data arrays directly to `analyze_array()`.

In [ ]:
analyzer = RingDownAnalyzer()
result = analyzer.analyze_array(t=t, data=data)

print("Analysis results (numpy arrays):")
print(f"  Sampling frequency: {result['fs']:.2f} Hz")
print(f"  Estimated tau: {result['tau_est']:.6f} s (true: {signal.tau:.6f})")
print(f"  NLS frequency: {result['f_nls']:.9f} Hz (true: {f0})")
print(f"  DFT frequency: {result['f_dft']:.9f} Hz")
print(f"  Q (NLS): {result['Q_nls']:.1f}")
print(f"  CRLB std: {result['crlb_std_f']:.6e} Hz")

## Analyze with Data and Sampling Rate Only

If you only have the signal and sampling frequency, time is inferred as `t = np.arange(len(data)) / fs`.

In [ ]:
result2 = analyzer.analyze_array(data=data, fs=fs)
print("Analysis from (data, fs) - same results:")
print(f"  f_nls: {result2['f_nls']:.9f} Hz")
print(f"  Q_nls: {result2['Q_nls']:.9f} Hz")
print(f"  tau_est: {result2['tau_est']:.6f} s")

## Analyze with Pandas Series

Pandas Series are converted to numpy arrays internally.

In [ ]:
series = pd.Series(data)
result3 = analyzer.analyze_array(t=t, data=series)
print("Analysis from pandas Series:")
print(f"  f_nls: {result3['f_nls']:.9f} Hz")
print(f"  Q_nls: {result3['Q_nls']:.9f} Hz")
print(f"  tau_est: {result3['tau_est']:.6f} s")

## Analyze with Pandas DataFrame

For DataFrames, specify column names or indices for time and signal.

In [ ]:
df = pd.DataFrame({"time_s": t, "phase_cycles": data})
result4 = analyzer.analyze_array(
    data=df,
    time_col="time_s",
    data_col="phase_cycles",
)
print("Analysis from pandas DataFrame:")
print(f"  f_nls: {result4['f_nls']:.9f} Hz")
print(f"  Q_nls: {result4['Q_nls']:.9f} Hz")
print(f"  tau_est: {result4['tau_est']:.6f} s")

## Visualize Results

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# Time series: original vs cropped
ax.plot(result["t"], result["data"], "b-", alpha=0.5, label="Full data")
ax.plot(
    result["t_crop"],
    result["data_cropped"],
    "r-",
    alpha=0.8,
    label="Cropped (≤1×τ)",
)
ax.axvline(
    result["tau_est"],
    color="g",
    linestyle="--",
    label=f"τ = {result['tau_est']:.3f} s",
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Phase (cycles)")
ax.set_title("Synthetic Noisy Ring-Down Signal")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zoom: first τ only, x-axis = sample index
t_max_plot = result["tau_est"]
mask = result["t"] <= t_max_plot
n = np.arange(len(result["t"]))[mask]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(n, result["data"][mask], "b-", alpha=0.6)
ax.axvline(
    int(result["tau_est"] * result["fs"]),
    color="g",
    linestyle="--",
    label=f"τ ≈ sample {int(result['tau_est'] * result['fs'])}",
)
ax.set_xlabel("Sample index")
ax.set_ylabel("Phase (cycles)")
ax.set_title("Zoom: First τ (sample index)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()